In [6]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

def Data_Loader(batch_size=32, use_augmentation=False):
    transform = [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]

    train_transform = transform.copy()
    if use_augmentation:
        train_transform.insert(0, transforms.RandomHorizontalFlip())
        train_transform.insert(0, transforms.RandomRotation(15))
    train_transform = transforms.Compose(train_transform)
    test_transform = transforms.Compose(transform)

    train_set = torchvision.datasets.OxfordIIITPet(root='./data', split='trainval', download=True, transform=train_transform)
    test_set = torchvision.datasets.OxfordIIITPet(root='./data', split='test', download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, train_set.classes

In [7]:
import torch.nn as nn
import torch.nn.functional as F

class PetNet(nn.Module):
    def __init__(self, num_classes=37, use_batchnorm=False, dropout_p=0.0):
        super(PetNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) if use_batchnorm else nn.Identity()

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(dropout_p)

        self.fc1 = nn.Linear(64*32*32, 256)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [8]:
import torch
from tqdm import tqdm

def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    pbar = tqdm(train_loader, desc='Training', unit='batch')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        pbar.set_postfix(loss=loss.item())

        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix(loss=loss.item(), acc=100. * correct / ((pbar.n + 1) * train_loader.batch_size))
    return total_loss / len(train_loader), 100. * correct / len(train_loader.dataset)

def validate_model(model, test_loader, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / len(test_loader.dataset)

In [9]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from data_loader import Data_Loader
from model import PetNet
from train import train_model, validate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

experiments = [
    {"name": "Basic", "bn": False, "dropout": 0.0, "aug": False, "lr": 0.001},
    {"name": "Regularization", "bn": True, "dropout": 0.3, "aug": False, "lr": 0.001},
    {"name": "Maxed Out", "bn": True, "dropout": 0.5, "aug": True, "lr": 0.0005},
]

results = {}

for exp in experiments:
    print(f"\n--- Experiment {exp['name']} ---")
    train_loader, test_loader, _ = Data_Loader(use_augmentation=exp['aug'])
    model = PetNet(use_batchnorm=exp['bn'], dropout_p=exp['dropout']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=exp['lr'])
    criterion = nn.CrossEntropyLoss()

    history = []
    for epoch in range(1, 6):
        loss, acc = train_model(model, train_loader, criterion, optimizer, device)
        test_acc = validate_model(model, test_loader, device)
        history.append(test_acc)
        print(f"\n--- Epoch {epoch}: Train Accuracy: {acc:.4f}% -- Test Accuracy: {test_acc:.4f}% -- Loss: {loss:.4f} ---")
    results[exp['name']] = history

plt.figure(figsize=(10, 6))
for name, hist in results.items():
    plt.plot(hist, label=name)
plt.title("Comparing Experiments - Test Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


--- Experiment Basic ---

--- Epoch 1: Train Accuracy: 3.4511% -- Test Accuracy: 6.4130% -- Loss: 3.6887 ---

--- Epoch 2: Train Accuracy: 7.2283% -- Test Accuracy: 9.6739% -- Loss: 3.4457 ---

--- Epoch 3: Train Accuracy: 13.4239% -- Test Accuracy: 28.6957% -- Loss: 3.2039 ---

--- Epoch 4: Train Accuracy: 29.7826% -- Test Accuracy: 57.5543% -- Loss: 2.5387 ---

--- Epoch 5: Train Accuracy: 61.1413% -- Test Accuracy: 91.3043% -- Loss: 1.4196 ---

--- Experiment Regularization ---

--- Epoch 1: Train Accuracy: 2.4457% -- Test Accuracy: 2.5815% -- Loss: 3.7469 ---

--- Epoch 2: Train Accuracy: 2.2283% -- Test Accuracy: 2.7174% -- Loss: 3.6118 ---

--- Epoch 3: Train Accuracy: 2.7174% -- Test Accuracy: 2.7717% -- Loss: 3.6117 ---

--- Epoch 4: Train Accuracy: 2.9348% -- Test Accuracy: 4.4565% -- Loss: 3.6036 ---


KeyboardInterrupt: 